# Contents

1. Load XGBoost and NN models
2. Explore model performance
3. Ensemble model objective & method
4. Ensemble model implementation
5. Evaluation

In [14]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_recall_curve, roc_curve, log_loss,
    precision_score, recall_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV

import xgboost as xgb
from tensorflow.keras.models import load_model as keras_load_model

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer


RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Load XGBoost and NN models

In [15]:
nn_keras_path = "../models/NN/NN_1to5_all.keras"
nn_model = keras_load_model(nn_keras_path)

In [16]:
xgb_json_path = "../models/XGB/best_xgb_1to5_all.json"
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(xgb_json_path)

## Explore model performance

Based on our validation data, we evaluate the performance of both the XGBoost and NN models. As the two models have different input types, we have separate data preparation pipelines, although the underlying data used is the same.

In [17]:
df_train = pd.read_csv('../data/FEwithMerchants/FE_train_downsampled_1to5_with_merchants.csv')
df_val = pd.read_csv('../data/FEwithMerchants/FE_validation_with_merchants.csv')
df_test = pd.read_csv('../data/FEwithMerchants/FE_test_with_merchants.csv')

In [18]:
df_train_xgb = df_train.copy()
df_train_nn = df_train.copy()

df_val_xgb = df_val.copy()
df_val_nn = df_val.copy()

df_test_xgb = df_test.copy()
df_test_nn = df_test.copy()

### Data preparation for XGBoost

In [19]:
print("Converting categorical columns ...")
categorical_cols = [
    'type', 'hourOfDay', 'dayOfWeek', 'dayOfWeekName', 'transaction_sequence', 'isFlaggedFraud', 'typeHighValueFlag', 'amountBucket', 
]

for col in categorical_cols:
    # Combine categories from both train, val and test, to avoid unseen categories during inference 
    combined_cats = pd.Series(
        pd.concat([df_train_xgb[col], df_val_xgb[col], df_test_xgb[col]], axis=0).dropna().unique()
    )
    categories = sorted(combined_cats.unique())

    df_train_xgb[col] = pd.Categorical(df_train[col], categories=categories)
    df_val_xgb[col]   = pd.Categorical(df_val[col], categories=categories)
    df_test_xgb[col]  = pd.Categorical(df_test[col], categories=categories)

Converting categorical columns ...


In [20]:
# selected features for all
selected_features = [
    # transaction-level features
    'step', 'type', 'hourOfDay', 'day', 'amountLog', 'dayOfWeek', 'amount', 
    'oldbalanceOrg', 'oldbalanceDest',
    'amount_to_oldbalanceOrg',
     
                
    # account-level features
    'meanSent', 
    'totalSent', 'stdSent', 'numSent', 
    'totalReceived',  'numReceived', 
    'stdReceived', 'meanReceived', 
    'maxAmountReceived', 'stdAmountReceived', 'std_to_mean_ratio', 
    'avgAmountToDest', 
    'pctForwarded24h', 'pairFrequency', 'pctUniqueDest', 'pctUniqueOrig', 

    # transaction pattern features 
    'transaction_sequence', 'sequence_frequency', 'transactionRecency', 'typeHighValueFlag', 
    'is_early_transaction',  'sequence_count',
    'is_transfer_cashout', 'is_cashin_transfer', 'is_cashout_transfer', 'is_cashin_transfer_cashout',
    'is_transfer_transfer', 'is_first_transfer', 'is_cashin_cashout', 

    # centrality features
    'sender_btwn', 'receiver_btwn', 'btwn_diff', 
    'sender_outdeg_amt', 'sender_indeg_amt', 'receiver_outdeg_amt', 'receiver_indeg_amt',
    'sender_outdeg_cnt', 'sender_indeg_cnt', 'receiver_outdeg_cnt', 'receiver_indeg_cnt', 
    'outdeg_amt_diff', 'indeg_amt_diff',
    'outdeg_cnt_diff', 'indeg_cnt_diff' 
]

In [21]:
print("Processing training set ...")
Xtrain_xgb, ytrain_xgb = df_train_xgb[selected_features].copy(), df_train_xgb["isFraud"].astype(int)

print("Processing validation set ...")
Xval_xgb, yval_xgb = df_val_xgb[selected_features].copy(), df_val_xgb["isFraud"].astype(int)

print("Processing test set ...")
Xtest_xgb, ytest_xgb = df_test_xgb[selected_features].copy(), df_test_xgb["isFraud"].astype(int)

print("Done!")

Processing training set ...
Processing validation set ...
Processing test set ...
Done!


### Data preparation for NN

In [22]:
selected_features = [
    # transaction-level features
    'step', 'type', 'day', 'amountLog', 'amount_to_oldbalanceOrg','dayOfWeek','hourOfDay',
    # 'amount',         
                
    # account-level features
    'meanSent', 'numUniqueDest', 'numUniqueOrig',
    'transaction_sequence',
    'totalSent', 'stdSent', 'numSent', 
    'totalReceived', 'meanReceived', 'stdReceived', 'numReceived', 'maxAmountReceived', 
    'stdAmountReceived', 'avgAmountToDest', 'std_to_mean_ratio', 
    'pctForwarded24h', 'oldbalanceOrg', 'oldbalanceDest',

    # transaction pattern features
    'transactionRecency', 'is_early_transaction',  'sequence_frequency', 'sequence_count',
    'is_transfer_cashout', 'is_cashin_transfer', 'is_cashout_transfer', 'is_cashin_transfer_cashout',
    'is_transfer_transfer', 'is_first_transfer', 'is_cashin_cashout', 

    # aggregated risk signals
    'typeHighValueFlag', 

    # centrality features
    "receiver_indeg_amt",
    "sender_outdeg_amt","sender_indeg_amt", "receiver_outdeg_amt",
    "sender_outdeg_cnt","sender_indeg_cnt", "receiver_outdeg_cnt","receiver_indeg_cnt",
    "outdeg_amt_diff","indeg_amt_diff", "outdeg_cnt_diff","indeg_cnt_diff", 'sender_btwn',
    'receiver_btwn', 'btwn_diff'
]

In [23]:
missing_features = [col for col in df_train_nn.columns if col not in selected_features]
print("Missing features:", missing_features)

Missing features: ['amount', 'nameOrig', 'nameDest', 'isFlaggedFraud', 'isFraud', 'avgAmountPerType', 'p95AmountPerType', 'amountBucket', 'fraudProbabilityByAmountBin', 'dayOfWeekName', 'pairFrequency', 'pctUniqueDest', 'pctUniqueOrig']


In [24]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
columns_to_encode = ['type', 'transaction_sequence', 'typeHighValueFlag']

encoder.fit(df_train_nn[columns_to_encode])

encoded_train = encoder.transform(df_train_nn[columns_to_encode])
encoded_test = encoder.transform(df_test_nn[columns_to_encode])
encoded_val = encoder.transform(df_val_nn[columns_to_encode])


encoded_columns = encoder.get_feature_names_out(columns_to_encode).tolist()


encoded_df_train = pd.DataFrame(encoded_train, columns=encoded_columns, index=df_train_nn.index)
encoded_df_test = pd.DataFrame(encoded_test, columns=encoded_columns, index=df_test_nn.index)
encoded_df_val = pd.DataFrame(encoded_val, columns=encoded_columns, index=df_val_nn.index)

df_train_nn = pd.concat([df_train_nn, encoded_df_train], axis=1)
df_test_nn = pd.concat([df_test_nn, encoded_df_test], axis=1)
df_val_nn = pd.concat([df_val_nn, encoded_df_val], axis=1)

In [25]:
# remove original categorical names from selected_features and append encoded names
selected_features = [c for c in selected_features if c not in columns_to_encode] + encoded_columns

# drop original categorical columns from dataframes to avoid string columns later
df_train_nn = df_train_nn.drop(columns=columns_to_encode, errors='ignore')
df_test_nn  = df_test_nn.drop(columns=columns_to_encode, errors='ignore')
df_val_nn   = df_val_nn.drop(columns=columns_to_encode, errors='ignore')

In [26]:
# Prepare data
X_train_final = df_train_nn[selected_features].copy()
y_train_final = df_train_nn['isFraud'].copy()
X_test_final = df_test_nn[selected_features].copy()
y_test_final = df_test_nn['isFraud'].copy()
X_val_final = df_val_nn[selected_features].copy()
y_val_final = df_val_nn['isFraud'].copy()

In [27]:
print("\nScaling features...")

numeric_cols = X_train_final.select_dtypes(include=['number']).columns
scaler = StandardScaler()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols)
])

imbalanced_X_train_scaled = preprocessor.fit_transform(X_train_final)
imbalanced_X_test_scaled = preprocessor.transform(X_test_final)
imbalanced_X_val_scaled = preprocessor.transform(X_val_final)

# Handle NaN/Inf
Xtrain_nn = np.nan_to_num(imbalanced_X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
Xtest_nn = np.nan_to_num(imbalanced_X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)
Xval_nn = np.nan_to_num(imbalanced_X_val_scaled, nan=0.0, posinf=0.0, neginf=0.0)

ytrain_nn = y_train_final.values.astype(int)
ytest_nn = y_test_final.values.astype(int)
yval_nn = y_val_final.values.astype(int)

print(f"Train: {Xtrain_nn.shape}, Fraud rate: {ytrain_nn.mean():.4f}")
print(f"Val: {Xtest_nn.shape}, Fraud rate: {ytest_nn.mean():.4f}")
print(f"Test: {Xval_nn.shape}, Fraud rate: {yval_nn.mean():.4f}")


Scaling features...


/Users/charlesgoek/Documents/Work/DSA4263 Project/DSA4263/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/Users/charlesgoek/Documents/Work/DSA4263 Project/DSA4263/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/Users/charlesgoek/Documents/Work/DSA4263 Project/DSA4263/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


Train: (34494, 61), Fraud rate: 0.1667
Val: (954393, 61), Fraud rate: 0.0013
Test: (954393, 61), Fraud rate: 0.0013


### XGBoost and NN model performance
We look at performance on validation data to see where gaps in each model may lie.

In [28]:
def keras_proba_binary(model, X: np.ndarray) -> np.ndarray:
    """
    Returns P(y=1) from a Keras model.
    Accepts:
      - sigmoid output: shape (n, 1)
      - softmax output: shape (n, 2)
    """
    p = np.asarray(model.predict(X, verbose=0))
    if p.ndim == 2 and p.shape[1] == 1:  # sigmoid
        return p.ravel()
    if p.ndim == 2 and p.shape[1] == 2:  # softmax
        return p[:, 1]
    if p.ndim == 1:                      # already (n,)
        return p
    raise ValueError(f"Unexpected Keras output shape: {p.shape}")

def xgb_proba_binary(model: xgb.XGBClassifier, X: np.ndarray) -> np.ndarray:
    p = model.predict_proba(X)
    return p[:, 1] if p.ndim == 2 else p

def metrics_from_proba(y_true, p, thr=0.5):
    pred = (p >= thr).astype(int)   
    return {
        "roc_auc": roc_auc_score(y_true, p),
        "pr_auc":  average_precision_score(y_true, p),
        "log_loss": log_loss(y_true, np.clip(p, 1e-6, 1-1e-6)),
        "f1@thr":  f1_score(y_true, pred, zero_division=0),
        "precision@thr": precision_score(y_true, pred, zero_division=0),
        "recall@thr":    recall_score(y_true, pred, zero_division=0),
        "threshold": float(thr),
    }

def tune_threshold_fbeta(y_true, p, beta=1.0):
    """
    Find threshold maximizing F-beta on precision-recall curve.
    Returns (best_thr, best_fbeta).
    """
    prec, rec, thrs = precision_recall_curve(y_true, p)
    fbeta = (1 + beta**2) * (prec * rec) / np.clip(beta**2 * prec + rec, 1e-12, None)
    fbeta = fbeta[1:]  # align with thrs length
    idx = int(np.argmax(fbeta))
    return float(thrs[idx]), float(fbeta[idx])

Previously established F1-optimised threshold for XGBoost and NN are 0.3104 and 0.8404 respectively. We use these thresholds to evaluate their individual performance.

In [29]:
thr_xgb = 0.3104  
p_xgb_val = xgb_proba_binary(xgb_model, Xval_xgb)
base_xgb_val = metrics_from_proba(yval_xgb, p_xgb_val, thr=thr_xgb)

print("Validation — Base XGB:", base_xgb_val)

Validation — Base XGB: {'roc_auc': 0.9995801499861084, 'pr_auc': 0.9694191926896643, 'log_loss': 0.06970521786961689, 'f1@thr': 0.3424905870868777, 'precision@thr': 0.20676881629904023, 'recall@thr': 0.9967532467532467, 'threshold': 0.3104}


In [30]:
thr_nn = 0.8404 
p_nn_val  = keras_proba_binary(nn_model, Xval_nn)
base_nn_val  = metrics_from_proba(yval_nn, p_nn_val, thr=thr_nn)

print("Validation — Base NN :", base_nn_val)

Validation — Base NN : {'roc_auc': 0.971608747037557, 'pr_auc': 0.5456718123733278, 'log_loss': 0.006944222562162484, 'f1@thr': 0.5425264217413186, 'precision@thr': 0.713907284768212, 'recall@thr': 0.4375, 'threshold': 0.8404}


In [ ]:
thr_xgb = 0.3104 
thr_nn  = 0.8404

pred_xgb_val = (p_xgb_val >= thr_xgb).astype(int)
pred_nn_val  = (p_nn_val  >= thr_nn).astype(int)

err_xgb = pred_xgb_val != yval_xgb
err_nn  = pred_nn_val  != yval_nn

both_wrong     = np.mean(err_xgb & err_nn)
only_xgb_wrong = np.mean(err_xgb & ~err_nn)
only_nn_wrong  = np.mean(~err_xgb & err_nn)

print("Fraction both wrong     :", both_wrong)
print("Fraction only XGB wrong :", only_xgb_wrong)
print("Fraction only NN wrong  :", only_nn_wrong)

Fraction both wrong     : 0.00013411665844154346
Fraction only XGB wrong : 0.004806196189619999
Fraction only NN wrong  : 0.0008183211737722301


At their tuned thresholds, the neural network achieved a higher F1 score than XGBoost. While XGBoost reached extremely high recall (≈0.997) but low precision (≈0.21), the neural network showed the opposite trend—lower recall (≈ 0.44) but much higher precision (≈ 0.71). Using their respective F1-optimized thresholds, both models were wrong on only 0.0134% of cases. Of all cases, the neural network was wrong while XGBoost was correct on 0.0818%, while XGBoost was wrong and the neural network was correct on 0.481%. These results indicate that although XGBoost dominates overall with a PR-AUC of ≈0.969 compared to the neural network which had PR-AUC of ≈0.546, each model captures slightly different aspects of the data.

## Ensemble model objective and method


Since the two models make different types of errors, combining them could help reduce those remaining mistakes. In particular, a stacking ensemble could learn to rely more on XGBoost for broad detection and adjust toward the neural network in cases where its higher precision adds value. The goal is to capture these few complementary cases and improve overall balance between recall and precision.

Stacking is an ensemble technique where predictions from multiple base models are used as inputs to a meta-model, which learns how to best combine them. In our case, the meta-model (Logistic Regression) learns a set of coefficients that minimize prediction loss on the validation set, effectively determining how much to trust each base model’s output. 

Unlike simple weighted averaging, these weights are not manually fixed but are learned through optimization, resulting in a globally optimal combination of the base models based on their validation performance.

## Ensemble model implementation

In [33]:
p_xgb_test = xgb_proba_binary(xgb_model, Xtest_xgb)
p_nn_test  = keras_proba_binary(nn_model, Xtest_nn)

In [35]:
# Stacking inputs: just the two probabilities
P_val  = np.column_stack([p_xgb_val,  p_nn_val])
P_test = np.column_stack([p_xgb_test, p_nn_test])

meta = LogisticRegression(
    C=1.0,
    solver="lbfgs",
    max_iter=200,
    random_state=42
)
meta.fit(P_val, yval_xgb)

print("Meta coefficients (xgb, nn):", meta.coef_)
print("Meta intercept:", meta.intercept_)

Meta coefficients (xgb, nn): [[9.0267177 5.6325139]]
Meta intercept: [-10.16407826]


In [40]:
# After: meta.fit(P_val, yval)
p_meta_val = meta.predict_proba(P_val)[:, 1]

# beta = 1.0 for F1, >1.0 to emphasize recall
thr_meta, fbest_meta = tune_threshold_fbeta(yval_xgb, p_meta_val, beta=1.0)

meta_val_metrics = metrics_from_proba(yval_xgb, p_meta_val, thr=thr_meta)
meta_val_metrics.update({
    "fbeta_best": fbest_meta,
    "tuned_threshold": thr_meta,
})

print("Validation — Stacking (tuned):", meta_val_metrics)

p_meta_test = meta.predict_proba(P_test)[:, 1]

meta_test_metrics = metrics_from_proba(ytest_xgb, p_meta_test, thr=thr_meta)
meta_test_metrics.update({
    "threshold": thr_meta,
})

print("Test — Stacking (tuned):", meta_test_metrics)

Validation — Stacking (tuned): {'roc_auc': 0.999469755603491, 'pr_auc': 0.9116435634820795, 'log_loss': 0.001239123317286951, 'f1@thr': 0.8308605341246291, 'precision@thr': 0.7650273224043715, 'recall@thr': 0.9090909090909091, 'threshold': 0.23329459955403634, 'fbeta_best': 0.831168831168831, 'tuned_threshold': 0.23329459955403634}
Test — Stacking (tuned): {'roc_auc': 0.999128674170784, 'pr_auc': 0.9026090156661235, 'log_loss': 0.0013014168952738285, 'f1@thr': 0.8211320754716981, 'precision@thr': 0.767277856135402, 'recall@thr': 0.8831168831168831, 'threshold': 0.23329459955403634}


In [42]:
xgb_test_metrics = metrics_from_proba(ytest_xgb, p_xgb_test, thr=thr_xgb)
nn_test_metrics = metrics_from_proba(ytest_nn, p_nn_test, thr=thr_nn)

p_stack_test = meta.predict_proba(P_test)[:, 1]
stack_test_metrics = metrics_from_proba(ytest_xgb, p_stack_test, thr=thr_meta)

print("Test — XGB only:",  xgb_test_metrics)
print("Test — NN only:", nn_test_metrics)
print("Test — Stacking:", stack_test_metrics)

Test — XGB only: {'roc_auc': 0.9992230465909624, 'pr_auc': 0.9599844804400313, 'log_loss': 0.06974042215215039, 'f1@thr': 0.33913764510779437, 'precision@thr': 0.20436375749500332, 'recall@thr': 0.9959415584415584, 'threshold': 0.3104}
Test — NN only: {'roc_auc': 0.9655016210961049, 'pr_auc': 0.5158130645575626, 'log_loss': 0.007168839703889285, 'f1@thr': 0.514344262295082, 'precision@thr': 0.6972222222222222, 'recall@thr': 0.4074675324675325, 'threshold': 0.8404}
Test — Stacking: {'roc_auc': 0.999128674170784, 'pr_auc': 0.9026090156661235, 'log_loss': 0.0013014168952738285, 'f1@thr': 0.8211320754716981, 'precision@thr': 0.767277856135402, 'recall@thr': 0.8831168831168831, 'threshold': 0.23329459955403634}


In [44]:
# Using tuned thresholds
thr_xgb = 0.3104
thr_stack = 0.23329459955403634

pred_xgb_test = (p_xgb_test >= thr_xgb).astype(int)
pred_stack_test = (p_meta_test >= thr_stack).astype(int)

# Compute confusion matrix for each
from sklearn.metrics import confusion_matrix

cm_xgb = confusion_matrix(ytest_xgb, pred_xgb_test)
cm_stack = confusion_matrix(ytest_xgb, pred_stack_test)

print("XGB confusion matrix:\n", cm_xgb)
print("Stacking confusion matrix:\n", cm_stack)

XGB confusion matrix:
 [[948384   4777]
 [     5   1227]]
Stacking confusion matrix:
 [[952831    330]
 [   144   1088]]


In [45]:
tn_xgb, fp_xgb, fn_xgb, tp_xgb = cm_xgb.ravel()
tn_stack, fp_stack, fn_stack, tp_stack = cm_stack.ravel()

print(f"XGB — FP: {fp_xgb}, FN: {fn_xgb}")
print(f"Stack — FP: {fp_stack}, FN: {fn_stack}")

XGB — FP: 4777, FN: 5
Stack — FP: 330, FN: 144


## Evaluation

On the test set, XGBoost continued to perform strongly with a PR-AUC of 0.9600, showing excellent overall discrimination. However, its high recall (≈ 0.996) came at the cost of low precision (≈ 0.204), indicating that it flagged nearly all fraud cases but also produced many false positives. The neural network, in contrast, had lower overall scores (ROC-AUC ≈ 0.9655, PR-AUC ≈ 0.516) but achieved much higher precision (≈ 0.697) with lower recall (≈ 0.407), confirming its more conservative nature.

The stacked ensemble achieved a PR-AUC of 0.9026, with a strong F1 score of 0.821. It balanced both precision (≈ 0.767) and recall (≈ 0.883) more effectively than either base model alone. Although its overall PR-AUC was marginally lower than XGBoost’s, the substantial improvement in F1 demonstrates that stacking successfully combined the high recall of XGBoost with the higher precision of the neural network, leading to a better overall trade-off between detecting fraud and limiting false alarms.

The confusion matrices reveal a clear trade-off between the two models. XGBoost demonstrates extremely high recall, missing only 5 fraud cases out of more than 1,200, but it does so at the cost of generating 4,777 false positives (legitimate transactions incorrectly flagged as fraud). In contrast, the stacked ensemble is more conservative as it reduces false positives dramatically from 4,777 to just 330, though this comes with an increase in false negatives from 5 to 144, meaning it overlooks more actual fraud cases.

In essence, XGBoost prioritizes catching nearly every fraudulent transaction, even if that means flagging many normal ones, while the ensemble model focuses on precision, flagging fewer cases but being more confident when it does. Which model works better really depends on what matters more in practice. If missing a fraud case is costly, XGBoost is the safer option since it catches almost everything. But if false alerts are time-consuming or hurt user experience (like triggering unnecessary reviews or frustrating legitimate users) the stacked ensemble offers a more balanced and practical choice.

## Appendix: weighted average ensemble

In [38]:
# Search w in [0,1] for: p = w*p_xgb + (1-w)*p_nn
ws = np.linspace(0, 1, 101)
best = {"w": None, "score": -np.inf, "metric": "pr_auc"}

for w in ws:
    p_val = w * p_xgb_val + (1 - w) * p_nn_val
    score = average_precision_score(yval_xgb, p_val)  # use PR AUC for imbalanced tasks
    if score > best["score"]:
        best.update({"w": float(w), "score": float(score)})

w_best = best["w"]
p_wavg_val = w_best * p_xgb_val + (1 - w_best) * p_nn_val

# Tune decision threshold for chosen policy (F-beta)
thr_wavg, fbest = tune_threshold_fbeta(yval_xgb, p_wavg_val, beta=1.0)  # set beta>1 to favor recall
wavg_val_metrics = metrics_from_proba(yval_xgb, p_wavg_val, thr=thr_wavg)
wavg_val_metrics.update({"weight_xgb": w_best, "weight_nn": 1 - w_best, "fbeta_best": fbest})

print(f"weight = {w_best}")

print("Validation — Weighted Avg:", wavg_val_metrics)

weight = 1.0
Validation — Weighted Avg: {'roc_auc': 0.9995801499861084, 'pr_auc': 0.9694191926896643, 'log_loss': 0.06970521786961689, 'f1@thr': 0.9150435142975549, 'precision@thr': 0.934801016088061, 'recall@thr': 0.8961038961038961, 'threshold': 0.9830347299575806, 'weight_xgb': 1.0, 'weight_nn': 0.0, 'fbeta_best': 0.9154228855721394}
